# Databricks AI Functions — `ai_extract` and `ai_classify`

**Problem:** vendor invoices and policy PDFs arrive in 5 languages. Humans open each one and key in the details.

**Databricks answer:** AI is built into SQL. No model to train, no infrastructure to run:

* `ai_parse_document` → reads a PDF into structured elements
* `ai_extract` → turns the document into **structured data** (invoice id, vendor, amount...)
* `ai_classify` → assigns **business labels** with confidence and rationale

All 8 invoices and 3 policy PDFs live in the `retail_documents` Volume.

In [0]:
%sql
-- Step 0: what documents are we dealing with?
SELECT
  path,
  length(content) AS bytes
FROM READ_FILES(
  '/Volumes/retail_ai_prod/unstructured_docs/retail_documents/invoices/',
  format => 'binaryFile'
)

path,bytes
dbfs:/Volumes/retail_ai_prod/unstructured_docs/retail_documents/invoices/invoice_05_english_gst_invoice.pdf,2431
dbfs:/Volumes/retail_ai_prod/unstructured_docs/retail_documents/invoices/invoice_03_french_two_column.pdf,2398
dbfs:/Volumes/retail_ai_prod/unstructured_docs/retail_documents/invoices/invoice_07_portuguese_receipt.pdf,2384
dbfs:/Volumes/retail_ai_prod/unstructured_docs/retail_documents/invoices/invoice_02_spanish_compact_receipt.pdf,2380
dbfs:/Volumes/retail_ai_prod/unstructured_docs/retail_documents/invoices/invoice_01_english_classic_table.pdf,2375
dbfs:/Volumes/retail_ai_prod/unstructured_docs/retail_documents/invoices/invoice_04_german_minimal.pdf,2373
dbfs:/Volumes/retail_ai_prod/unstructured_docs/retail_documents/invoices/invoice_06_english_handwritten_style.pdf,2356
dbfs:/Volumes/retail_ai_prod/unstructured_docs/retail_documents/invoices/invoice_08_english_rtl_like.pdf,2356


### Step 1 — Extract structured data from every invoice

`ai_parse_document` reads each PDF, then `ai_extract` returns typed fields — invoice id, vendor, date, amount, currency — with a confidence score, in any of the 5 languages.

In [0]:
%sql
-- Step 1: PARSE each PDF, then EXTRACT structured invoice data from it.
-- ai_parse_document reads the PDF; ai_extract turns it into typed fields (v2.1 adds confidence scores).
WITH parsed_docs AS (
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
  FROM READ_FILES(
    '/Volumes/retail_ai_prod/unstructured_docs/retail_documents/invoices/',
    format => 'binaryFile'
  )
)
SELECT
  path,
  ai_extract(
    parsed_content,
    '{
      "invoice_id":    {"type": "string"},
      "vendor_name":   {"type": "string"},
      "invoice_date":  {"type": "string", "description": "Invoice date in YYYY-MM-DD format"},
      "total_amount":  {"type": "number"},
      "currency":      {"type": "string"}
    }',
    MAP(
      'version', '2.1',
      'instructions', 'These are retail vendor invoices in English, Spanish, French, German and Portuguese.',
      'enableConfidenceScores', 'true'
    )
  ) AS invoice_data
FROM parsed_docs

### Step 2 — Land it as a governed Unity Catalog table

The unstructured world becomes a queryable table: `retail_ai_prod.unstructured_docs.invoice_extracted` — governed, permissioned, joinable with sales data.

In [0]:
%sql
-- Step 2: land the extracted fields as a GOVERNED Unity Catalog table.
-- No more manual keying-in: the unstructured world just became queryable.
CREATE OR REPLACE TABLE retail_ai_prod.unstructured_docs.invoice_extracted AS
WITH parsed_docs AS (
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
  FROM READ_FILES(
    '/Volumes/retail_ai_prod/unstructured_docs/retail_documents/invoices/',
    format => 'binaryFile'
  )
),
extracted AS (
  SELECT
    path,
    ai_extract(
      parsed_content,
      '{
        "invoice_id":    {"type": "string"},
        "vendor_name":   {"type": "string"},
        "invoice_date":  {"type": "string", "description": "Invoice date in YYYY-MM-DD format"},
        "total_amount":  {"type": "number"},
        "currency":      {"type": "string"}
      }',
      MAP('version', '2.1')
    ) AS invoice_data
  FROM parsed_docs
)
SELECT
  regexp_replace(path, '.*/', '')                    AS file_name,
  invoice_data:response:invoice_id:value::STRING     AS invoice_id,
  invoice_data:response:vendor_name:value::STRING    AS vendor_name,
  invoice_data:response:invoice_date:value::STRING   AS invoice_date,
  invoice_data:response:total_amount:value::DOUBLE   AS total_amount,
  invoice_data:response:currency:value::STRING       AS currency
FROM extracted;

SELECT * FROM retail_ai_prod.unstructured_docs.invoice_extracted

### Step 3 — Classify the policy documents

`ai_classify` routes each policy PDF to the team that owns it — customer service, legal or product support — with a confidence score and a rationale you can show an auditor.

In [0]:
%sql
-- Step 3: CLASSIFY the policy documents — which team owns each one?
-- Labels carry descriptions; v2.1 returns confidence + a rationale you can show an auditor.
WITH parsed_policies AS (
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
  FROM READ_FILES(
    '/Volumes/retail_ai_prod/unstructured_docs/retail_documents/policies/',
    format => 'binaryFile'
  )
)
SELECT
  regexp_replace(path, '.*/', '') AS file_name,
  ai_classify(
    parsed_content,
    '{
      "customer_service": "Return, refund and exchange rules for shoppers",
      "legal":             "Vendor contracts, supply agreements, liability clauses",
      "product_support":  "Product warranties and repair coverage"
    }',
    MAP(
      'version', '2.1',
      'instructions', 'Route each retail policy document to the team that owns it.',
      'enableConfidenceScores', 'true',
      'enableRationales', 'true'
    )
  ) AS routing
FROM parsed_policies

## What just happened

* 8 invoices in 5 languages → a **queryable UC table** in 2 lines of SQL
* 3 policy PDFs → **routed to the owning team** with confidence + rationale
* Zero infrastructure, zero model training — the AI is part of the SQL engine
* Everything ran under Unity Catalog governance on the same data platform as the sales tables